<a href="https://colab.research.google.com/github/Pritesh934/05-Alignment-Bioinformatics-Assignment/blob/main/Gut_to_Soil_QIIME2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gut-to-soil microbiome analysis

Name: **[enter name]** &nbsp; Date: **[enter date]**

Follow the [Gut-to-soil tutorial, release 2026.10](https://gut-to-soil-tutorial.readthedocs.io/en/2026.10/). Use its **Command Line** tabs.

Your goal is to turn paired-end 16S reads into a community profile, compare environments, and connect the analysis to machine learning.

### How to use this notebook
1. Run the three **Provided setup** sections unchanged.
2. Work through the tutorial and notebook together, one cell at a time. Do **not** use Run all while placeholders remain.
3. For each numbered task, copy the indicated `qiime ...` command from the tutorial's **Command Line** tab and paste it **between** the triple quotes, replacing `PASTE_TUTORIAL_COMMAND_HERE`. Include its option lines and backslashes. Do not paste Markdown fences, `$`, `!`, `cd`, or the Python API example.
4. Keep tutorial filenames. The provided `tutorial_step` function runs the command in the installed QIIME environment and correct directory. You do not need to activate an environment or add `!`.
5. Use the visualization selector after each section to download `.qzv` outputs; open them in [QIIME 2 View](https://view.qiime2.org/). Write observations in the response cells.
6. Save your notebook with responses and download the results ZIP at the end.

One **"Worked example"** is included before the first task. Blank cells stop with a clear message rather than running an answer for you.


### Colab adjustments handled for you
The adapter prints the final command before running it. It limits DADA2 threads, taxonomy-training batch size, and taxonomy-classification workers/batches. It saves the classifier annotation to a separate file and uses that annotated copy downstream. Provenance paths are adjusted to this notebook's folder. All biological settings remain those in your pasted command. Reference artifacts and input reads are provided by download helpers so the exercise focuses on analytical commands.


## Provided setup A — Settings and file helpers

Keep the release fixed. The configuration records downloaded-file checksums and the installed environment. The release environment is pinned by its published specification; its URL is not a content-addressed lockfile. Downloads and generated results live on the Colab VM until you export them.

If you change an analysis parameter, rerun that cell and every downstream cell. The runner detects changed inputs/commands and recomputes outputs. The student tasks compute DADA2; the recovery section provides separately labeled reference plots if needed.


In [ ]:
from pathlib import Path
import os, sys, json, shlex, shutil, subprocess, hashlib, time, urllib.request
import tarfile, zipfile, platform, html

RELEASE = "2026.10"
DADA2_MODE = "compute"             # "compute" or "checkpoint"
REFERENCE_SOURCE = "tutorial-artifacts"  # or "rebuild" (download from GTDB)
THREADS = 2
TRAINING_CHUNK_SIZE = 200          # tutorial uses 2000; smaller lowers transient RAM
SAMPLING_DEPTH = 96                # shallow tutorial subset, not a research recommendation
RAREFACTIONS = 10
SEED = 42
assert DADA2_MODE in {"compute", "checkpoint"}
assert REFERENCE_SOURCE in {"tutorial-artifacts", "rebuild"}
assert platform.system() == "Linux" and platform.machine() in {"x86_64", "AMD64"}

ROOT = Path('/content/gut_to_soil_student')
WORK = ROOT / 'analysis'
LOGS = ROOT / 'logs'
for p in (ROOT, WORK, LOGS): p.mkdir(parents=True, exist_ok=True)
MM = ROOT / 'bin' / 'micromamba'
PREFIX = ROOT / ('qiime-' + RELEASE)
BASE = f'https://gut-to-soil-tutorial.readthedocs.io/en/{RELEASE}/data/gut-to-soil/'
ENV_URL = f'https://raw.githubusercontent.com/qiime2/distributions/refs/heads/dev/{RELEASE}/qiime2/released/rachis-qiime2-linux-64-conda.yml'
RUN_ENV = os.environ.copy()
# Prevent notebook-side Python settings from leaking into the QIIME process.
for key in ('PYTHONPATH', 'PYTHONHOME'): RUN_ENV.pop(key, None)
RUN_ENV.update(MAMBA_ROOT_PREFIX=str(ROOT / 'mamba'), PYTHONNOUSERSITE='1',
               MPLBACKEND='Agg', OMP_NUM_THREADS=str(THREADS),
               OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda: f.read(1024*1024), b''): h.update(b)
    return h.hexdigest()

def record(name, item):
    p = ROOT / name
    data = json.loads(p.read_text()) if p.exists() else []
    data.append(dict(time=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), **item))
    p.write_text(json.dumps(data, indent=2))

def download(url, dest, *, refresh=False):
    dest = Path(dest); dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size and not refresh:
        return dest
    temp = dest.with_name(dest.name + '.part')
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, headers={'User-Agent':'GutToSoilTeachingNotebook/1.0'})
            with urllib.request.urlopen(req, timeout=120) as response, temp.open('wb') as out:
                shutil.copyfileobj(response, out)
            if not temp.stat().st_size: raise ValueError('Empty download')
            if dest.suffix in {'.qza', '.qzv', '.zip'}:
                if not zipfile.is_zipfile(temp): raise ValueError('Expected ZIP/QIIME artifact; got another file type')
                with zipfile.ZipFile(temp) as z:
                    bad = z.testzip()
                    if bad: raise ValueError('Corrupt ZIP member: '+bad)
            elif dest.suffix in {'.tsv', '.yml'}:
                beginning = temp.read_text()[:200].lower()
                if '<html' in beginning or '<!doctype' in beginning:
                    raise ValueError('Received an HTML page instead of data')
            temp.replace(dest)
            record('downloads.json', dict(url=url, path=str(dest.relative_to(ROOT)),
                   bytes=dest.stat().st_size, sha256=sha256(dest)))
            print('Downloaded', dest.name, f'({dest.stat().st_size/1e6:.1f} MB)')
            return dest
        except Exception:
            temp.unlink(missing_ok=True)
            if attempt == 2: raise
            time.sleep(2)

def fetch(name, dest=None):
    return download(BASE + name, WORK / (dest or name))

def run(argv, log_name='command', cwd=None):
    argv = [str(a) for a in argv]
    print('$', shlex.join(argv), flush=True)
    log = LOGS / (log_name + '.log')
    started = time.monotonic()
    with log.open('w') as f:
        p = subprocess.Popen(argv, cwd=cwd or WORK, env=RUN_ENV,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
        try:
            for line in p.stdout:
                print(line, end='', flush=True); f.write(line); f.flush()
            rc = p.wait()
        except BaseException:
            p.terminate()
            try: p.wait(timeout=10)
            except subprocess.TimeoutExpired: p.kill(); p.wait()
            raise
    record('runs.json', dict(command=argv, returncode=rc,
           seconds=round(time.monotonic()-started, 2), log=str(log.relative_to(ROOT))))
    if rc:
        raise RuntimeError(f'Command failed (exit {rc}). Read {log}. No checkpoint was substituted.')

def qcmd(args): return [MM, 'run', '-p', PREFIX, 'qiime', *args]

mem = dict(line.split(':',1) for line in Path('/proc/meminfo').read_text().splitlines())
print('Available RAM:', round(int(mem['MemAvailable'].split()[0])/1024**2,1), 'GiB')
print('Free disk:', round(shutil.disk_usage(ROOT).free/1024**3,1), 'GiB')
print('Notebook Python:', sys.executable, '(will remain unchanged)')


## Provided setup B — Installation

This uses the official release environment with micromamba as its Conda-compatible installer. It does not replace Colab's Python. Installation may be lengthy; inspect the live output. If it fails, keep the installation log rather than adding arbitrary `pip install` fixes. The initial free-space check is a conservative classroom guard, not an official minimum.


In [ ]:
if not MM.exists():
    package = download('https://micro.mamba.pm/api/micromamba/linux-64/latest', ROOT/'micromamba.tar.bz2')
    with tarfile.open(package) as tar:
        member = tar.getmember('bin/micromamba')
        MM.parent.mkdir(exist_ok=True)
        with tar.extractfile(member) as src, MM.open('wb') as dst:
            shutil.copyfileobj(src, dst)
    MM.chmod(0o755)
run([MM, '--version'], 'micromamba-version')
env_file = download(ENV_URL, ROOT/f'qiime-{RELEASE}.yml')
ready = PREFIX / '.notebook-install-complete'
if not ready.exists():
    if shutil.disk_usage(ROOT).free < 20*1024**3:
        raise RuntimeError('Less than 20 GiB free. Start a fresh runtime or free space before installation.')
    run([MM, 'create', '-y', '-p', PREFIX, '-f', env_file], 'installation')
    run(qcmd(['info']), 'qiime-info')
    ready.write_text(sha256(env_file))
else:
    assert ready.read_text() == sha256(env_file), 'Environment specification changed; use a fresh runtime.'
    run(qcmd(['info']), 'qiime-info')
run([MM, 'list', '-p', PREFIX, '--explicit'], 'environment-explicit')
print('Installation passed qiime info. Next: test the actual tutorial actions.')


## Provided setup C — Compatibility checks and runner

This checks release-specific command flags before the analysis starts. A successful preflight verifies the CLI, not the analysis itself. Each step writes a success record only after all expected outputs exist. Changed input bytes or changed commands invalidate cached results. Partial outputs are regenerated on retry.


In [ ]:
checks = {
 ('dada2','denoise-paired'): ['--o-base-transition-stats','--p-n-threads'],
 ('feature-table','summarize'): ['--o-summary','--o-sample-frequencies','--m-metadata-file'],
 ('feature-table','tabulate-seqs'): ['--i-taxonomy'],
 ('feature-classifier','fit-classifier-naive-bayes'): ['--p-classify--chunk-size'],
 ('feature-classifier','classify-sklearn'): ['--p-reads-per-batch','--p-n-jobs'],
 ('boots','kmer-diversity'): ['--p-n','--p-no-replacement'],
 ('composition','ancombc2'): ['--p-fixed-effects-formula','--o-ancombc2-output'],
 ('composition','da-barplot'): ['--i-data'],
 ('rescript','get-gtdb-data'): ['--p-db-type'],
 ('tools','annotation-create'): ['--annotation-type'],
 ('tools','replay-provenance'): ['--in-fp','--out-fp'],
}
problems=[]
for action, flags in checks.items():
    p = subprocess.run([str(x) for x in qcmd([*action,'--help'])], env=RUN_ENV,
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    (LOGS/('help-'+'-'.join(action)+'.log')).write_text(p.stdout)
    missing=[f for f in flags if f not in p.stdout]
    if p.returncode or missing: problems.append((' '.join(action),missing,p.returncode))
if problems: raise RuntimeError(f'Release/API mismatch: {problems}. See help logs; do not mix tutorial releases.')
print('All tutorial API checks passed.')

def digest_path(p):
    p=Path(p)
    if p.is_file(): return sha256(p)
    if p.is_dir():
        return hashlib.sha256(json.dumps([(str(f.relative_to(p)),sha256(f))
             for f in sorted(p.rglob('*')) if f.is_file()]).encode()).hexdigest()
    return None

def step(name, command, outputs):
    argv=shlex.split(command)
    assert argv[0]=='qiime'
    paths=[WORK/o for o in outputs]
    for p in paths:
        assert p.resolve().is_relative_to(WORK.resolve()) and p.resolve()!=WORK.resolve()
    inputs={}
    for token in argv[1:]:
        candidate=WORK/token
        # Named collection arguments use label:path.
        if not candidate.exists() and ':' in token: candidate=WORK/token.split(':',1)[1]
        if candidate.exists() and candidate not in paths:
            inputs[str(candidate.relative_to(WORK))]=digest_path(candidate)
    signature=hashlib.sha256(json.dumps([command,inputs,RELEASE],sort_keys=True).encode()).hexdigest()
    marker=LOGS/('step-'+hashlib.sha256(name.encode()).hexdigest()[:12]+'.json')
    old=json.loads(marker.read_text()) if marker.exists() else {}
    observed={o:digest_path(WORK/o) for o in outputs}
    if old.get('signature')==signature and all(observed.values()) and old.get('outputs')==observed:
        print('Already completed:',name); return
    marker.unlink(missing_ok=True)
    for p in paths:
        if p.is_dir(): shutil.rmtree(p)
        elif p.exists(): p.unlink()
    run(qcmd(argv[1:]), 'step-'+hashlib.sha256(name.encode()).hexdigest()[:12])
    observed={o:digest_path(WORK/o) for o in outputs}
    if not all(observed.values()): raise RuntimeError(f'Missing outputs for {name}: {observed}')
    marker.write_text(json.dumps(dict(name=name,signature=signature,outputs=observed),indent=2))

def view_outputs():
    from IPython.display import display, HTML
    import ipywidgets as w
    paths=sorted(WORK.rglob('*.qzv'))
    if not paths: print('No visualizations yet.'); return
    choices=w.Dropdown(options=[(str(p.relative_to(WORK)),str(p)) for p in paths],
                       description='Result:',layout=w.Layout(width='90%'))
    button=w.Button(description='Download selected .qzv')
    def selected(_):
        from google.colab import files
        files.download(choices.value)
    button.on_click(selected)
    display(HTML('<p>Download a visualization, then open it at <a href="https://view.qiime2.org/" target="_blank">QIIME 2 View</a>. Inspect its provenance as well as its plots.</p>'))
    display(choices,button)


## Provided setup D — Paste-in adapter
Run once. This accepts a **single QIIME command** per task. It recognizes the requested outputs, applies the Colab adjustments listed above, and passes the command to the tested runner. It does not fill in missing commands or invent results.


In [ ]:
def tutorial_step(task, command):
    if 'PASTE_TUTORIAL_COMMAND_HERE' in command or not command.strip():
        raise ValueError(f'{task}: paste the indicated tutorial command between the triple quotes first.')
    command = command.replace('\\\n', '')
    args = shlex.split(command)
    if not args or args[0] != 'qiime' or args.count('qiime') != 1:
        raise ValueError('Paste exactly one command beginning with qiime, without !, $, cd, or code fences.')
    if any(x in args for x in [';', '&&', '|']):
        raise ValueError('Paste one command only, without shell operators.')
    def option(flag, value):
        if flag in args:
            j=args.index(flag)
            if j+1==len(args): raise ValueError(f'Missing value after {flag}')
            args[j+1]=str(value)
        else: args.extend([flag,str(value)])
    action=tuple(args[1:3])
    if action == ('dada2','denoise-paired'): option('--p-n-threads', THREADS)
    if action == ('feature-classifier','fit-classifier-naive-bayes'):
        option('--p-classify--chunk-size',TRAINING_CHUNK_SIZE)
    if action == ('feature-classifier','classify-sklearn'):
        option('--p-n-jobs',1); option('--p-reads-per-batch',100)
        j=args.index('--i-classifier')+1
        if args[j]=='suboptimal-16S-rRNA-classifier.qza': args[j]='teaching-classifier.qza'
    if action == ('tools','annotation-create'):
        option('--output-path','teaching-classifier.qza')
    if action == ('tools','replay-provenance'):
        option('--in-fp',WORK); option('--out-fp',ROOT/'g2s-replayed.bash')
        run(qcmd(args[1:]),'student-provenance')
        return
    outputs=[]
    for i,arg in enumerate(args):
        if arg.startswith('--o-') or arg in {'--output-dir','--output-path'}:
            if i+1==len(args): raise ValueError(f'Missing filename after {arg}')
            outputs.append(args[i+1])
    if not outputs: raise ValueError('No output flags found. Paste the entire command, including output options.')
    print('Command to run:',shlex.join(args))
    step(task,shlex.join(args),outputs)


## Worked example — Metadata
The tutorial's **Reviewing sample metadata and running a QIIME 2 command** section creates a metadata visualization. Notice that `--m-input-file` identifies metadata and `--o-visualization` names an output. Everything between the triple quotes is a normal tutorial command.

The downloads below are provided. Metadata describes each sample; `demux.qza` contains the already imported paired-end reads.


In [ ]:
fetch('sample-metadata.tsv')
fetch('demux.qza')


In [ ]:
tutorial_step("Worked example: metadata", r"""
qiime metadata tabulate \
  --m-input-file sample-metadata.tsv \
  --o-visualization sample-metadata.qzv
""")
view_outputs()


## 4. Get the tutorial reads and metadata

**Find in the tutorial:** Summarize demultiplexed sequences.

Before executing, identify what each input contains and predict what this step will produce.


### Task 1: Read quality

Find the command beginning **`qiime demux summarize`**.

**Expected output(s):** `demux.qzv`.


In [ ]:
tutorial_step('Task 1: Read quality', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


In [ ]:
view_outputs()


**Pause and explain:** What are the observations in the metadata? Which column describes sample environment? Record one observation from the read-quality plot.

**Your response:**

[Write your observations and interpretation here.]


## 5. DADA2: reads → ASVs and counts

**Find in the tutorial:** Sequence quality control and feature table construction.

Before executing, identify what each input contains and predict what this step will produce.


### Task 2: DADA2

Find the command beginning **`qiime dada2 denoise-paired`**.

**Expected output(s):** `asv-seqs.qza`, `asv-table.qza`, `denoising-stats.qza`, `base-transition-stats.qza`.


In [ ]:
tutorial_step('Task 2: DADA2', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


### Task 3: DADA2 statistics

Find the command beginning **`qiime metadata tabulate`**.

**Expected output(s):** `denoising-stats.qzv`.


In [ ]:
tutorial_step('Task 3: DADA2 statistics', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


### Task 4: Combined metadata

Find the command beginning **`qiime metadata tabulate`**.

**Expected output(s):** `sample-metadata-w-dada2-stats.qzv`.


In [ ]:
tutorial_step('Task 4: Combined metadata', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


In [ ]:
view_outputs()


**Pause and explain:** Which processing stage lost the most reads for one sample you chose? Explain the distinction between an ASV, a read, and a species.

**Your response:**

[Write your observations and interpretation here.]


## 6. Summarize the count table

**Find in the tutorial:** Feature table and feature data summaries.

Before executing, identify what each input contains and predict what this step will produce.


### Task 5: 6. Summarize the count table

Find the command beginning **`qiime feature-table summarize`**.

**Expected output(s):** `asv-table.qzv`, `sample-frequencies.qza`, `asv-frequencies.qza`.


In [ ]:
tutorial_step('Task 5: 6. Summarize the count table', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


### Task 6: ASV sequences

Find the command beginning **`qiime feature-table tabulate-seqs`**.

**Expected output(s):** `asv-seqs.qzv`.


In [ ]:
tutorial_step('Task 6: ASV sequences', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


In [ ]:
view_outputs()


**Pause and explain:** How many samples and ASVs remain? Find the most frequent ASV and record a sequence-based observation about it.

**Your response:**

[Write your observations and interpretation here.]


## 7. Filter rare features

**Find in the tutorial:** Filtering features from a feature table.

Before executing, identify what each input contains and predict what this step will produce.


### Task 7: 7. Filter rare features

Find the command beginning **`qiime feature-table filter-features`**.

**Expected output(s):** `asv-table-ms2.qza`.


In [ ]:
tutorial_step('Task 7: 7. Filter rare features', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")

### Task 8: Filter sequences

Find the command beginning **`qiime feature-table filter-seqs`**.

**Expected output(s):** `asv-seqs-ms2.qza`.


In [ ]:
tutorial_step('Task 8: Filter sequences', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


### Task 9: Filtered table summary

Find the command beginning **`qiime feature-table summarize`**.

**Expected output(s):** `asv-table-ms2.qzv`, `sample-frequencies-ms2.qza`, `asv-frequencies-ms2.qza`.


In [ ]:
tutorial_step('Task 9: Filtered table summary', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


In [ ]:
view_outputs()


**Pause and explain:** How many ASVs were removed? Explain why excluding single-sample features might change an estimate of richness.

**Your response:**

[Write your observations and interpretation here.]


## 8. Reference sequences and taxonomy labels

**Find in the tutorial:** Taxonomic annotation → Training a taxonomic classifier.

Before executing, identify what each input contains and predict what this step will produce.


**Provided reference download:** These are the release-matched GTDB 202.0 artifacts used in the working instructor notebook. Skip the tutorial’s `rescript get-gtdb-data` command; continue with the reference-sequence summary. The downstream model is trained by you.


In [ ]:
fetch('reference-sequences.qza')
fetch('reference-taxonomy.qza')


### Task 10: Reference summary

Find the command beginning **`qiime feature-table tabulate-seqs`**.

**Expected output(s):** `reference-taxonomy.qzv`.


In [ ]:
tutorial_step('Task 10: Reference summary', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


In [ ]:
view_outputs()


**Pause and explain:** What are the training examples and their labels for taxonomy classification? How does the reference set limit the predictions?

**Your response:**

[Write your observations and interpretation here.]


## 9. Train a naïve Bayes taxonomy classifier

**Find in the tutorial:** Taxonomic annotation → Training a taxonomic classifier / Annotating an Artifact or Visualization.

Before executing, identify what each input contains and predict what this step will produce.


### Task 11: Train taxonomy classifier

Find the command beginning **`qiime feature-classifier fit-classifier-naive-bayes`**.

**Expected output(s):** `suboptimal-16S-rRNA-classifier.qza`.


In [ ]:
tutorial_step('Task 11: Train taxonomy classifier', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


### Task 12: Annotate classifier

Find the command beginning **`qiime tools annotation-create`**.

**Expected output(s):** `suboptimal-16S-rRNA-classifier.qza`.


Copy the tutorial’s complete annotation command. The adapter saves its result as `suboptimal-16S-rRNA-classifier.qza`, and uses that copy in the next task.


In [ ]:
tutorial_step('Task 12: Annotate classifier', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


In [ ]:
view_outputs()


**Pause and explain:** What do k-mer features encode? Explain why changing chunk size is different from changing the number of feature buckets.

**Your response:**

[Write your observations and interpretation here.]


## 10. Assign taxonomy

**Find in the tutorial:** Apply our taxonomy classifier.

Before executing, identify what each input contains and predict what this step will produce.


### Task 13: 10. Assign taxonomy

Find the command beginning **`qiime feature-classifier classify-sklearn`**.

**Expected output(s):** `taxonomy.qza`.


In [ ]:
tutorial_step('Task 13: 10. Assign taxonomy', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


### Task 14: Annotated ASVs

Find the command beginning **`qiime feature-table tabulate-seqs`**.

**Expected output(s):** `asv-seqs-ms2.qzv`.


In [ ]:
tutorial_step('Task 14: Annotated ASVs', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


In [ ]:
view_outputs()


**Pause and explain:** Choose one ASV. Record its taxonomy and deepest assigned rank. Why might an assignment stop above species level?

**Your response:**

[Write your observations and interpretation here.]


## 11. Community composition

**Find in the tutorial:** Taxonomic analysis.

Before executing, identify what each input contains and predict what this step will produce.


### Task 15: 11. Community composition

Find the command beginning **`qiime taxa barplot`**.

**Expected output(s):** `taxa-bar-plots.qzv`.


In [ ]:
tutorial_step('Task 15: 11. Community composition', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


In [ ]:
view_outputs()


**Pause and explain:** Compare two focal sample types at two taxonomic levels. Explain why relative abundance does not establish absolute abundance.

**Your response:**

[Write your observations and interpretation here.]


## 12. Rarefaction

**Find in the tutorial:** Selecting an even sampling depth.

Before executing, identify what each input contains and predict what this step will produce.


### Task 16: 12. Rarefaction

Find the command beginning **`qiime diversity alpha-rarefaction`**.

**Expected output(s):** `alpha-rarefaction.qzv`.


In [ ]:
tutorial_step('Task 16: 12. Rarefaction', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


In [ ]:
view_outputs()


**Pause and explain:** Compare sampling depths using both diversity curves and sample retention. Justify a depth, including its tradeoff.

**Your response:**

[Write your observations and interpretation here.]


## 13. Alpha diversity, beta diversity, and ordination

**Find in the tutorial:** Running the kmer-diversity Pipeline.

Before executing, identify what each input contains and predict what this step will produce.


### Task 17: Kmer diversity

Find the command beginning **`qiime boots kmer-diversity`**.

**Expected output(s):** `boots-kmer-diversity`.


In [ ]:
tutorial_step('Task 17: Kmer diversity', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


In [ ]:
view_outputs()


**Pause and explain:** Which categories separate in PCoA? Explain how PCoA differs from PCA. Does separation alone demonstrate predictive accuracy?

**Your response:**

[Write your observations and interpretation here.]


## 14. Restrict the differential-abundance comparison

**Find in the tutorial:** Differential abundance testing with ANCOM-BC2.

Before executing, identify what each input contains and predict what this step will produce.


### Task 18: 14. Restrict the differential-abundance comparison

Find the command beginning **`qiime feature-table filter-samples`**.

**Expected output(s):** `asv-table-ms2-dominant-sample-types.qza`.


In [ ]:
tutorial_step('Task 18: 14. Restrict the differential-abundance comparison', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


In [ ]:
view_outputs()


**Pause and explain:** Which three sample types are retained? Why is this count table not rarefied before ANCOM-BC2?

**Your response:**

[Write your observations and interpretation here.]


## 15. ANCOM-BC2 at ASV level

**Find in the tutorial:** Differential abundance testing with ANCOM-BC2.

Before executing, identify what each input contains and predict what this step will produce.


### Task 19: 15. ANCOM-BC2 at ASV level

Find the command beginning **`qiime composition ancombc2`**.

**Expected output(s):** `ancombc2-results.qza`.


In [ ]:
tutorial_step('Task 19: 15. ANCOM-BC2 at ASV level', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


### Task 20: ASV differential-abundance plot

Find the command beginning **`qiime composition da-barplot`**.

**Expected output(s):** `ancombc2-barplot.qzv`.


In [ ]:
tutorial_step('Task 20: ASV differential-abundance plot', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


In [ ]:
view_outputs()


**Pause and explain:** Choose one ASV comparison. Report direction, effect size, and adjusted evidence. What does the reference level mean?

**Your response:**

[Write your observations and interpretation here.]


## 16. Repeat at genus level

**Find in the tutorial:** Exercise 15: Perform ANCOM-BC2 on genera, instead of ASVs.

Before executing, identify what each input contains and predict what this step will produce.


### Task 21: 16. Repeat at genus level

Find the command beginning **`qiime taxa collapse`**.

**Expected output(s):** `genus-table-ms2-dominant-sample-types.qza`.


In [ ]:
tutorial_step('Task 21: 16. Repeat at genus level', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


### Task 22: Genus ANCOM-BC2

Find the command beginning **`qiime composition ancombc2`**.

**Expected output(s):** `genus-ancombc2-results.qza`.


In [ ]:
tutorial_step('Task 22: Genus ANCOM-BC2', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


### Task 23: Genus differential-abundance plot

Find the command beginning **`qiime composition da-barplot`**.

**Expected output(s):** `genus-ancombc2-barplot.qzv`.


In [ ]:
tutorial_step('Task 23: Genus differential-abundance plot', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


In [ ]:
view_outputs()


**Pause and explain:** Did aggregation to genus clarify or obscure a result? Explain why results can differ at ASV and genus levels.

**Your response:**

[Write your observations and interpretation here.]


## 17. Reconstruct provenance

**Find in the tutorial:** Ensuring bioinformatics reproducibility and adapting the tutorial workflow for your own data.

Before executing, identify what each input contains and predict what this step will produce.


### Task 24: Replay provenance

Paste only the `qiime tools replay-provenance` command. Skip `cd`: the adapter supplies the correct folders. Expected output: `g2s-replayed.bash`.


In [ ]:
tutorial_step('Task 24: Replay provenance', r"""
PASTE_TUTORIAL_COMMAND_HERE
""")


**Pause and explain:** Find the DADA2 trimming settings and diversity sampling depth in the replay script. Explain why it may contain commands you did not run yourself.

**Your response:**

[Write here.]


## Optional course extension — Sample-level machine learning
The official tutorial ends with provenance. The following **provided** extension connects your ASV table to the ML material from class. Run it after completing the tutorial, if directed. Read the explanation before running; predictions do not require the differential-abundance results.

**Predict first:** Which pair of sample types will be hardest to distinguish, and why?

**Your prediction:** [Write here.]


### Read before running the provided ML extension

This is an added course exercise, separate from the core tutorial. It runs in QIIME's environment so Colab does not need additional scientific packages installed.

**Features:** ASV counts from the unfiltered table. **Target:** SampleType for the three focal categories. The workflow makes a stratified 75/25 split, learns a prevalence filter on training data, uses a Hellinger transform (square roots of per-sample relative abundances), and fits a random forest. A majority-class baseline and shuffled-training-label control contextualize performance. The held-out set is for one evaluation, not repeated hyperparameter selection.

For ordination, PCA is fitted on transformed training data, then test samples are projected. PCA is for visualization here and is not an input to the classifier. ASV inference itself used all reads, as in the tutorial; this is an instructional evaluation within one sequencing study, not an independent external validation.

The script also reports bucket overlap and attempts grouped cross-validation. Missing bucket identifiers receive sample-specific groups, which **does not establish independence**. Shared donors, sites, or collection batches may require stronger grouping. Review the metadata before making claims about generalization.


In [ ]:
ml_script = ROOT / 'ml_extension.py'
ml_script.write_text('"""Teaching extension. Use --synthetic-test for a mechanics-only smoke test."""\nfrom pathlib import Path\nimport sys, json\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom sklearn.base import BaseEstimator, TransformerMixin\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.dummy import DummyClassifier\nfrom sklearn.decomposition import PCA\nfrom sklearn.model_selection import train_test_split, StratifiedGroupKFold, cross_validate\nfrom sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,\n                             ConfusionMatrixDisplay)\n\n\nclass PrevalenceHellinger(BaseEstimator, TransformerMixin):\n    def __init__(self, min_samples=2):\n        self.min_samples=min_samples\n\n    def fit(self, X, y=None):\n        X=np.asarray(X, dtype=float)\n        self.keep_=(X>0).sum(axis=0)>=self.min_samples\n        if not self.keep_.any(): raise ValueError(\'No features survive training-only prevalence filtering\')\n        return self\n\n    def transform(self, X):\n        X=np.asarray(X,dtype=float)[:,self.keep_]\n        sums=X.sum(axis=1,keepdims=True)\n        # Empty rows map to zeros; report these in the main evaluation.\n        return np.sqrt(np.divide(X,sums,out=np.zeros_like(X),where=sums>0))\n\n\ndef main(work, seed=42, synthetic=False):\n    out=work/\'ml\'; out.mkdir(parents=True,exist_ok=True)\n    labels=[\'Food Compost\',\'Human Excrement\',\'Human Excrement Compost\']\n    if synthetic:\n        rng=np.random.default_rng(seed)\n        y0=np.repeat(labels,20)\n        raw=rng.poisson(2,(60,30))\n        for k in range(3): raw[k*20:(k+1)*20,k*10:(k+1)*10]+=rng.poisson(4,(20,10))\n        X=pd.DataFrame(raw,index=[f\'s{i}\' for i in range(60)],columns=[f\'asv{i}\' for i in range(30)])\n        metadata=pd.DataFrame({\'SampleType\':y0,\'Bucket\':[str(i//3) for i in range(60)]},index=X.index)\n    else:\n        from biom import load_table\n        import zipfile, tempfile\n        with tempfile.TemporaryDirectory() as temp:\n            with zipfile.ZipFile(work/\'asv-table.qza\') as z:\n                names=[n for n in z.namelist() if n.endswith(\'/data/feature-table.biom\')]\n                if len(names)!=1: raise ValueError(\'Cannot locate feature table inside QIIME artifact\')\n                data=Path(temp)/\'feature-table.biom\'\n                with z.open(names[0]) as src, data.open(\'wb\') as dst:\n                    import shutil\n                    shutil.copyfileobj(src,dst)\n            table=load_table(str(data))\n            # BIOM stores features in rows; ML requires samples in rows.\n            X=table.to_dataframe(dense=True).T\n        metadata=pd.read_csv(work/\'sample-metadata.tsv\',sep=\'\\t\',index_col=0,dtype=str)\n        metadata=metadata.loc[~metadata.index.astype(str).str.startswith(\'#\')]\n        if not metadata.index.is_unique: raise ValueError(\'Duplicate sample identifiers in metadata\')\n        missing=X.index.difference(metadata.index)\n        if len(missing): raise ValueError(f\'Sample IDs missing metadata: {list(missing)}\')\n        metadata=metadata.reindex(X.index)\n    selected=metadata[\'SampleType\'].isin(labels) & (X.sum(axis=1)>0)\n    X=X.loc[selected].astype(float); metadata=metadata.loc[X.index]\n    y=metadata[\'SampleType\']\n    if y.nunique()!=3 or y.value_counts().min()<4:\n        raise ValueError(\'Insufficient samples per class for this teaching split\')\n    train,test=train_test_split(np.arange(len(X)),test_size=.25,stratify=y,random_state=seed)\n    Xtrain,Xtest=X.iloc[train],X.iloc[test]\n    ytrain,ytest=y.iloc[train],y.iloc[test]\n    split=pd.DataFrame({\'SampleType\':y,\'partition\':\'train\'})\n    split.loc[Xtest.index,\'partition\']=\'test\'; split.to_csv(out/\'split.tsv\',sep=\'\\t\')\n    X.to_csv(out/\'asv-counts.tsv\',sep=\'\\t\'); metadata.to_csv(out/\'metadata.tsv\',sep=\'\\t\')\n    def forest():\n        return make_pipeline(PrevalenceHellinger(2),RandomForestClassifier(\n            n_estimators=200,min_samples_leaf=2,class_weight=\'balanced\',random_state=seed,n_jobs=1))\n    models={\'Majority baseline\':DummyClassifier(strategy=\'most_frequent\'),\n            \'Random forest\':forest(),\'Shuffled training labels\':forest()}\n    lines=[\'SYNTHETIC MECHANICS TEST — NOT BIOLOGICAL RESULTS\' if synthetic else \'Gut-to-soil sample classification\',\n           f\'Samples: {X.shape[0]}; ASVs: {X.shape[1]}; train: {len(train)}; test: {len(test)}\',\n           \'Class counts:\\n\'+y.value_counts().to_string()]\n    fig,axes=plt.subplots(1,3,figsize=(16,5))\n    metrics=[]\n    for ax,(name,model) in zip(axes,models.items()):\n        target=np.random.default_rng(seed).permutation(ytrain.to_numpy()) if name.startswith(\'Shuffled\') else ytrain\n        model.fit(Xtrain,target); pred=model.predict(Xtest)\n        score=dict(model=name,accuracy=accuracy_score(ytest,pred),balanced_accuracy=balanced_accuracy_score(ytest,pred))\n        metrics.append(score)\n        lines.append(\'\\n\'+name+\'\\n\'+classification_report(ytest,pred,labels=labels,zero_division=0))\n        ConfusionMatrixDisplay.from_predictions(ytest,pred,labels=labels,\n             display_labels=[\'Food\',\'Excrement\',\'HE compost\'],ax=ax,colorbar=False,cmap=\'Blues\')\n        ax.set_title(name,fontsize=11)\n    fig.tight_layout(); fig.savefig(out/\'confusion-matrices.png\',dpi=160); plt.close(fig)\n    pd.DataFrame(metrics).to_csv(out/\'metrics.tsv\',sep=\'\\t\',index=False)\n    fitted=models[\'Random forest\']; transform=fitted.steps[0][1]\n    Htrain=transform.transform(Xtrain); Htest=transform.transform(Xtest)\n    lines.append(f\'ASVs retained using training data: {transform.keep_.sum()}\')\n    lines.append(f\'Test samples with no retained counts: {int((Htest.sum(axis=1)==0).sum())}\')\n    if min(Htrain.shape)>=2:\n        pca=PCA(n_components=2).fit(Htrain)\n        fig,ax=plt.subplots(figsize=(8,6))\n        for part,data,target,marker in [(\'train\',Htrain,ytrain,\'o\'),(\'test\',Htest,ytest,\'^\')]:\n            coords=pca.transform(data)\n            for label,color in zip(labels,[\'#238b45\',\'#9e3d78\',\'#cc7a00\']):\n                mask=target.to_numpy()==label\n                ax.scatter(coords[mask,0],coords[mask,1],c=color,marker=marker,alpha=.8,label=f\'{label} ({part})\')\n        ax.set(xlabel=f\'PC1 ({pca.explained_variance_ratio_[0]:.1%})\',ylabel=f\'PC2 ({pca.explained_variance_ratio_[1]:.1%})\',\n               title=\'PCA of Hellinger-transformed ASVs\\nTransform and PCA fitted on training samples\')\n        ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(out/\'pca.png\',dpi=160); plt.close(fig)\n    importance=pd.Series(fitted.steps[-1][1].feature_importances_,index=X.columns[transform.keep_]).sort_values(ascending=False)\n    importance.rename(\'impurity_importance\').to_csv(out/\'feature-importance.tsv\',sep=\'\\t\')\n    fig,ax=plt.subplots(figsize=(10,6))\n    importance.head(12).sort_values().plot.barh(ax=ax,color=\'#2a6f97\')\n    ax.set(xlabel=\'Random forest impurity importance (training fit)\',title=\'Predictive features: association, not causation\')\n    ax.tick_params(axis=\'y\',labelsize=7); fig.tight_layout(); fig.savefig(out/\'feature-importance.png\',dpi=160); plt.close(fig)\n    lines.append(\'Feature importance is training-derived impurity importance; correlated features can share importance.\')\n    if \'Bucket\' in metadata:\n        buckets=metadata[\'Bucket\'].fillna(\'\').str.strip()\n        missing=buckets.str.lower().isin([\'\',\'nan\',\'na\',\'n/a\',\'none\',\'not applicable\',\'not provided\',\'missing\'])\n        groups=pd.Series([\'sample:\'+str(i) if absent else \'bucket:\'+str(b)\n                          for i,b,absent in zip(X.index,buckets,missing)],index=X.index)\n        overlap=set(groups.iloc[train]) & set(groups.iloc[test])\n        lines.append(f\'Groups overlapping random training/test partitions: {sorted(overlap)}\')\n        lines.append(f\'Samples lacking bucket IDs: {int(missing.sum())}; independence of their sample-specific groups is unverified.\')\n        if groups.nunique()>=3:\n            cv=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=seed)\n            folds=list(cv.split(X,y,groups))\n            usable=all(set(y.iloc[a])==set(labels) and set(y.iloc[b])==set(labels) for a,b in folds)\n            if usable:\n                scores=cross_validate(forest(),X,y,cv=folds,scoring=\'balanced_accuracy\',error_score=\'raise\')\n                grouped=scores[\'test_score\']\n                lines.append(\'Separate 3-fold bucket-grouped CV balanced accuracies: \'+np.array2string(grouped,precision=3))\n                lines.append(f\'Mean: {grouped.mean():.3f}. This uses different folds/data from the single holdout; it is not a paired accuracy comparison.\')\n            else:\n                lines.append(\'Grouped CV skipped: a fold lacks a focal class. Review the study design and grouping before changing the split.\')\n        lines.append(\'Bucket grouping does not guarantee donor/site independence. Review shared donors and collection sources.\')\n    else:\n        lines.append(\'No Bucket field: no grouped evaluation. Inspect metadata before treating observations as independent.\')\n    lines.append(\'This is a small within-study teaching evaluation, not evidence of deployment-ready prediction.\')\n    text=\'\\n\'.join(lines); (out/\'evaluation.txt\').write_text(text); print(text)\n\n\nif __name__==\'__main__\':\n    if \'--synthetic-test\' in sys.argv:\n        main(Path(sys.argv[1]),synthetic=True)\n    else:\n        main(Path(sys.argv[1]),int(sys.argv[2]) if len(sys.argv)>2 else 42)\n')
run([MM, 'run', '-p', PREFIX, 'python', ml_script, str(WORK), str(SEED)], 'ml-extension')


In [ ]:
from IPython.display import display, Image
for name in ['pca.png','confusion-matrices.png','feature-importance.png']:
    path=WORK/'ml'/name
    if path.exists(): display(Image(filename=str(path)))
print((WORK/'ml'/'evaluation.txt').read_text())


### Student interpretation

1. Contrast sequence-level taxonomy classification with sample-level environment classification: observations, features, labels, and reference information.
2. Compare the random forest with the baseline and shuffled-label control. Which errors dominate? Why can accuracy alone be misleading?
3. Does the random split put samples from the same bucket on both sides? How does that affect the claim being evaluated?
4. Compare PCA and PCoA. Does visible separation guarantee accurate prediction on a new biological replicate?
5. Identify one influential ASV. Does importance establish causation? Compare predictive importance with differential abundance, explaining why they need not agree.
6. State two limitations: the small, shallow dataset; compositional features; study dependence; incomplete reference taxonomy; or another evidence-based concern.

**Optional experiment:** change one model setting only after stating a prediction. For genuine model selection, use cross-validation within the training set and reserve a new final test set. Repeatedly optimizing against the displayed test score contaminates that evaluation.

**Your responses:**

[Write here.]


## Export results before leaving Colab

This archive includes analysis artifacts, plots, scripts, logs, file checksums, and environment information. It excludes the installed environment, the large input read artifact, and reference/trained-classifier artifacts, which can be regenerated or downloaded again. Each downstream QIIME artifact retains its own provenance.


In [ ]:
archive=ROOT/'gut_to_soil_results.zip'
excluded={'demux.qza','reference-sequences.qza','reference-taxonomy.qza',
          'suboptimal-16S-rRNA-classifier.qza','teaching-classifier.qza'}
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for directory in [WORK,LOGS,ROOT/'reference-recovery']:
        if directory.exists():
            for p in directory.rglob('*'):
                if p.is_file() and p.name not in excluded:
                    z.write(p,p.relative_to(ROOT))
    for pattern in ['*.json','*.yml','*.py','*.bash']:
        for p in ROOT.glob(pattern): z.write(p,p.name)
print('Archive:',archive,round(archive.stat().st_size/1e6,1),'MB')
from google.colab import files
files.download(str(archive))


## Recovery: published visualizations without a working QIIME installation

Run section 1, then this cell if installation or a later computation prevents class from continuing. It downloads **published reference visualizations into a separate folder**, never into the computed analysis. These are not your computed results. Open them in QIIME 2 View. This recovery route supports interpretation, not a claim that the workflow ran successfully. It still requires working internet downloads.


In [ ]:
RECOVERY = False  # change to True only when intentionally using published results
if RECOVERY:
    recovery=ROOT/'reference-recovery'; recovery.mkdir(exist_ok=True)
    names=['demux.qzv','denoising-stats.qzv','asv-table-ms2.qzv','taxa-bar-plots.qzv',
           'alpha-rarefaction.qzv','boots-kmer-diversity/scatter_plot.qzv',
           'ancombc2-barplot.qzv','genus-ancombc2-barplot.qzv']
    failures=[]
    for name in names:
        try: download(BASE+name,recovery/name)
        except Exception as e: failures.append((name,str(e)))
    record('checkpoints.json',dict(step='Reference visualizations only',files=names,failures=failures))
    recovered=ROOT/'published_reference_visualizations.zip'
    with zipfile.ZipFile(recovered,'w') as z:
        for p in recovery.rglob('*.qzv'): z.write(p,p.relative_to(recovery))
    print('PUBLISHED REFERENCE RESULTS — not computed in this runtime. Failures:',failures)
    from google.colab import files
    files.download(str(recovered))
else:
    print('Recovery disabled. Your computed results are unchanged.')


## Final reflection and submission checklist

- [ ] I completed the tutorial commands in order and saved the notebook with outputs.
- [ ] I inspected the quality, taxonomy, ordination, and differential-abundance visualizations.
- [ ] I answered the interpretation prompts using my own results.
- [ ] I identified any downloaded reference results used instead of computed outputs.
- [ ] I downloaded the results ZIP and saved my completed notebook.

**Synthesis (one paragraph):** What evidence supports a difference among microbial communities in the three focal environments? Distinguish descriptive patterns, statistical association, and prediction. State one limitation of this dataset or analysis.

[Write here.]

### If something fails
A placeholder message means you have not pasted the command yet. An unknown option usually indicates the wrong tutorial release or an incomplete paste. Check that you used the Command Line tab, included all output flags, and retained filenames. A failed command prints its log location; preserve the error for your instructor. Do not run downstream commands until their inputs exist. Use the separately labeled recovery plots only if directed.

### Sources
[Gut-to-soil tutorial, QIIME 2 2026.10](https://gut-to-soil-tutorial.readthedocs.io/en/2026.10/), by Caporaso and colleagues, based on the study by Meilander and colleagues. The linked tutorial contains the full explanations and citations. This course scaffold preserves the working instructor notebook’s Colab setup and adds spaces for commands and interpretation. The optional sample-classification extension is course material, separate from the official tutorial.
